In [1]:
!pip install pandas scikit-learn matplotlib

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

In [3]:
data = []

labels = ["A","B","C"]

for i in range(150):

    data.append({
        "text_id": i,
        "text": f"Example text {i}",
        "label": np.random.choice(labels)
    })

df = pd.DataFrame(data)

df.head()

,text_id,text,label
0,0,Example text 0,A
1,1,Example text 1,B
2,2,Example text 2,C
3,3,Example text 3,B
4,4,Example text 4,C


In [4]:
seed = 42

train, temp = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=seed
)

val, test = train_test_split(
    temp,
    test_size=0.5,
    stratify=temp["label"],
    random_state=seed
)

print("Train:",len(train))
print("Val:",len(val))
print("Test:",len(test))

Train: 120
Val: 15
Test: 15


In [5]:
print("Train distribution")
print(train["label"].value_counts(normalize=True))

print("Val distribution")
print(val["label"].value_counts(normalize=True))

print("Test distribution")
print(test["label"].value_counts(normalize=True))

Train distribution
label
B    0.375
C    0.325
A    0.300
Name: proportion, dtype: float64
Val distribution
label
B    0.400000
C    0.333333
A    0.266667
Name: proportion, dtype: float64
Test distribution
label
A    0.333333
B    0.333333
C    0.333333
Name: proportion, dtype: float64


In [6]:
train["length"] = train["text"].apply(len)
val["length"] = val["text"].apply(len)
test["length"] = test["text"].apply(len)

print("Train avg length:",train["length"].mean())
print("Val avg length:",val["length"].mean())
print("Test avg length:",test["length"].mean())

Train avg length: 15.283333333333333
Val avg length: 15.2
Test avg length: 15.2


In [7]:
train_texts = set(train["text"])
val_texts = set(val["text"])
test_texts = set(test["text"])

print("train ∩ test:",len(train_texts & test_texts))
print("train ∩ val:",len(train_texts & val_texts))
print("val ∩ test:",len(val_texts & test_texts))

train ∩ test: 0
train ∩ val: 0
val ∩ test: 0


In [8]:
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(train["text"])
X_test = vectorizer.transform(test["text"])

similarity = cosine_similarity(X_train, X_test)

threshold = 0.95

pairs = np.where(similarity >= threshold)

print("Near duplicate pairs:",len(pairs[0]))

Near duplicate pairs: 120


In [9]:
patterns = ["label=","class=","category="]

def check_template(text):

    for p in patterns:
        if p in text.lower():
            return True

    return False

df["template_flag"] = df["text"].apply(check_template)

print("Template leakage rows:",df["template_flag"].sum())

Template leakage rows: 0


In [10]:
train["text_id"].to_csv("splits_train_ids.txt",index=False)
val["text_id"].to_csv("splits_val_ids.txt",index=False)
test["text_id"].to_csv("splits_test_ids.txt",index=False)